# Yellow Taxi Data Analysis for Whole 2009 Year

In [1]:
import polars as pl

## Step 1: Load Lazy Mode Through Polars Scan

In [2]:
path = r"C:\Users\ekadw\Documents\DATA\NY_Taxi\2009\yellow_taxi\*.parquet"
yellow_2009 = pl.scan_parquet(path)

## Step 2: Features Engineering

In [3]:
mapping = {
    "Credit": 0,
    "CREDIT": 0,
    "CASH": 1,
    "Cash": 1,
    "No Charge": 2,
    "Dispute": 3
}

In [4]:
yellow_2009 = (
    yellow_2009.select(["Trip_Pickup_DateTime", "Trip_Dropoff_DateTime", "Passenger_Count", "Trip_Distance", "Payment_Type",
                        "Fare_Amt", "Tip_Amt"])
               .filter((pl.col("Passenger_Count") >= 0) & (pl.col("Trip_Distance") >= 0) & (pl.col("Trip_Distance") <= 50) & 
                       (pl.col("Fare_Amt") >= 0) & (pl.col("Tip_Amt") >= 0))
               .with_columns(
                       pl.col("Payment_Type").replace(mapping))
               .with_columns(
                       pl.col("Payment_Type").cast(pl.Int64))
               .filter(
                       pl.col("Payment_Type") == 0)
               .with_columns([
                       pl.col("Trip_Pickup_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Trip_Dropoff_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Payment_Type").cast(pl.Int64)])
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds() / 86400)
                               .cast(pl.Int64)
                               .alias("Duration_Days"))
               .filter(
                       (pl.col("Duration_Days") == 0))
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds())
                               .cast(pl.Int64)
                               .alias("Duration_Seconds"))
               .with_columns(
                       pl.when(pl.col("Tip_Amt") <= 0.0).then(1)
                               .otherwise(0)
                               .alias("Tip_Category"))
               .select(["Passenger_Count", "Trip_Distance", "Fare_Amt", "Duration_Seconds", "Tip_Category"])
)

In [ ]:
yellow_2009 = yellow_2009.collect()

In [ ]:
len(yellow_2009)

In [ ]:
corr_df = yellow_2009.select(pl.all().cast(pl.Float64)).to_pandas().corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="coolwarm")
plt.show()

In [ ]:
yellow_2009["Tip_Category"].value_counts()

In [ ]:
y = yellow_2009.select('Tip_Category').to_numpy().ravel().astype(int)
X = yellow_2009.drop(['Tip_Category']).to_numpy()